In [1]:
import pandas as pd
import utilities as util

In [2]:
test = pd.read_parquet("/Users/masa6503/repos/swot-precip-validation/data/usgs_stage_data/usgs_stage_data_0_50.parquet")
stations = util.retrieve_stations()
test['usgs_site_no'] = test['monitoring_location_id'].astype(str).replace('USGS-', '', regex=True)
relevant_stations = stations[stations['usgs_site_no'].isin(test['usgs_site_no'])]

In [28]:
node_ids = relevant_stations['node_id'].unique()

In [29]:
fields = [
    # Index
    'node_id',
    'time_str',
    # Measurements
    'wse',
    'width',
    'area_total',
    # Quality -- to filter
    'node_q_b', # Not far or near swath (bits 13 + 14 set)
    'ice_clim_f', # Not ice (== 0)
    'xovr_cal_q', # Not bad xover (< 2)
    'p_dam_id', # Affected by dam
    # Possibly affected by rain
    'dark_frac', 'n_good_pix', # Were measurements made
    'lat', 'lon', 'lat_u', 'lon_u', # Location + uncertainties
    'rdr_sig0', 'rdr_sig0_u', 'rdr_pol', # Backscatter coefficient
    'dry_trop_c', 'wet_trop_c', 'iono_c', 'xovr_cal_c', # Corrections
    'p_wse', 'p_wse_var', 'p_width', 'p_wid_var', # Prior estimages
]

fetch_hydrocron_study = (
    lambda row: util.fetch_hydrocron_node(row, start_time, end_time, fields)
)

# Download only once
# if not swot_path.exists() or not use_cache:
#     # Threaded execution
#     swot_ts_df_list = []
#     with ThreadPoolExecutor(max_workers=10) as executor:
#         # Submit all tasks
#         futures = [executor.submit(fetch_hydrocron_study, row) 
#                 for _, row in site_gdf.iterrows()]
        
#         # Collect results with tqdm progress bar
#         for future in tqdm(as_completed(futures), total=len(futures)):
#             result = future.result()
#             if result is not None:
#                 swot_ts_df_list.append(result)

#     # Combine all results
#     swot_ts_df = pd.concat(swot_ts_df_list)

#     swot_ts_df.to_csv(swot_path)

# Download without checking for existence
# if not swot_path.exists() or not use_cache:
    # Threaded execution
swot_ts_df_list = []
with ThreadPoolExecutor(max_workers=10) as executor:
    # Submit all tasks
    futures = [executor.submit(fetch_hydrocron_study, row) 
            for _, row in site_gdf.iterrows()]
    
    # Collect results with tqdm progress bar
    for future in tqdm(as_completed(futures), total=len(futures)):
        result = future.result()
        if result is not None:
            swot_ts_df_list.append(result)

# Combine all results
swot_ts_df = pd.concat(swot_ts_df_list)

swot_ts_df.to_csv(swot_path)

array([72608900080021, 72608900010241, 72608700090611, 73112000680251,
       73112000790331, 73112001030371, 73112001110281, 73112000100341,
       73112000040265, 73114000880061, 73114000830581, 73114001360401,
       73114000760091, 73114001310121, 73114000680501, 73114000280771,
       73118000140011, 73120000600261, 73120000430621, 73120000380121,
       73120000270241, 73120000890221, 73120000190081, 73120000080161,
       73130000120681, 73130000120251, 73140800170901, 73140800120061,
       73140700090081, 73140500070021, 73140500060014, 73140400430191,
       73140400360101, 73140400170391, 73140400130241, 73140400020061,
       73140300100014, 73150600350341, 73150600330251, 73150600260511,
       73150600210481, 73150600190081, 73150600610801, 73150600130581,
       73150600120351, 73150600730024, 73160900030111, 73160900010014,
       73160800020401, 73160500190701])